In [ ]:
!pip install streamlit
!pip install pyngrok


In [ ]:
!pip install gradio --quiet

import gradio as gr
from PIL import Image
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

MODEL_PATH = "/content/Image_Detection_Model.h5"  # تأكد الملف موجود هنا
model = tf.keras.models.load_model(MODEL_PATH)

CLASS_NAMES = ["Glioma", "Meningioma", "Pituitary", "No Tumor"]
IMAGE_SIZE = (224, 224)

def preprocess_image(image: Image.Image):
    img = image.convert("RGB").resize(IMAGE_SIZE)
    arr = np.array(img).astype("float32") / 255.0
    arr = np.expand_dims(arr, axis=0)
    return arr

def predict_tumor(image):
    x = preprocess_image(image)
    preds = model.predict(x)[0]
    pred_idx = int(np.argmax(preds))
    pred_label = CLASS_NAMES[pred_idx]

    fig, ax = plt.subplots(figsize=(4,4))
    ax.pie(preds, labels=CLASS_NAMES, autopct="%1.1f%%", startangle=90, colors=plt.cm.Set3.colors)
    ax.axis("equal")

    return pred_label, fig

iface = gr.Interface(
    fn=predict_tumor,
    inputs=gr.Image(type="pil", label="Upload MRI Image"),
    outputs=[gr.Label(num_top_classes=1, label="Predicted Tumor Type"), gr.Plot(label="Prediction Probabilities")],
    title="🧠 Brain Tumor Classifier",
    description="Upload a brain MRI image and the model will predict the tumor type.",
    theme="default"
)

iface.launch(debug=True)
